In [1]:
import itertools
import os
import numpy as np
import pandas as pd
import sklearn
from sklearn import tree
import tensorflow as tf
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel,
    DotProduct,
    WhiteKernel,
)
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from tensorflow.keras.models import Model, load_model
from joblib import Parallel, delayed

pd.set_option("display.max_columns", None)
from TL_BaseModel import *
import multiprocessing

In [2]:
total_cores = multiprocessing.cpu_count()
cores_to_use = int(total_cores * 0.80)
cores_to_use = max(1, cores_to_use)
os.environ["LOKY_MAX_CPU_COUNT"] = str(cores_to_use)
print(f"Using {cores_to_use} out of {total_cores} available CPU cores (80%)")
intra_op_threads = int(cores_to_use * 0.75)
inter_op_threads = cores_to_use - intra_op_threads
tf.config.threading.set_inter_op_parallelism_threads(inter_op_threads)
tf.config.threading.set_intra_op_parallelism_threads(intra_op_threads)
print(
    f"TensorFlow thread configuration: {inter_op_threads} inter-op, {intra_op_threads} intra-op"
)
physical_devices = tf.config.list_physical_devices("CPU")

BATCH_SIZE = 1024
print("pd.__version__ = ", pd.__version__)
print("tf.__version__ = ", tf.__version__)
print("sklearn.__version__ = ", sklearn.__version__)
print("tf.config.list_physical_devices('GPU') = ", tf.config.list_physical_devices())
print("tf.test.is_built_with_cuda() = ", tf.test.is_built_with_cuda())
device = "cuda" if tf.test.is_built_with_cuda() else "cpu"
device = "cpu"
print("device = ", device)

Using 12 out of 16 available CPU cores (80%)
TensorFlow thread configuration: 3 inter-op, 9 intra-op
pd.__version__ =  2.2.3
tf.__version__ =  2.19.0
sklearn.__version__ =  1.6.1
tf.config.list_physical_devices('GPU') =  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
tf.test.is_built_with_cuda() =  False
device =  cpu


In [3]:
results_path = "./../../../Output_Results/"
tl_results_path = results_path + "2025_testing"
print("results_path", results_path)
print("tl_results_path", tl_results_path)

Architecture_GROUP_List = [
    "TL_All_Unfrozen",
]
Model_Type_list = [
    "K-NN",
    "LogisticRegression",
    "NaiveBayes",
    "SVM",
    "DecisionTree",
]
EVALUATION_METRICS = [
    "Weighted Accuracy",
    "Sensitivity/Recall",
    "Specificity",
    "Precision_class0",
    "Precision_class1",
    "Precision_avg",
    "F1_class0",
    "F1_class1",
    "F1_avg",
    "auc_roc_score",
    "False_Discovery_Rate",
    "False_Negative_Rate",
    "False_Omission_Rate",
    "False_Positive_Rate",
    "Jaccard",
]
number_of_splits = 3
augmentation_algo = "GammaSmoteBoost"
augmentation_algo = "ADKNN"
augmentation_algo = "SMOTEBoostCC"
augmentation_algo = "SDDSMOTEBoostCC"
bool_remove_target = False
num_pop_folds = 5
bool_drop_ABS = True
num_ind_folds = 3
POPULATION_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]
PATIENT_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]
target_grp = [
    "SB-001",
    "SB-003",
    "SB-008",
    "SB-011",
    "SB-012",
    "SB-022",
    "SB-023",
    "SB-025",
    "SB-028",
    "SB-037",
    "SB-043",
    "SB-056",
    "SB-059",
    "SB-060",
    "SB-070",
    "SB-071",
    "SB-073",
    "SB-078",
    "SB-079",
    "SB-080",
    "SB-082",
    "SB-083",
    "SB-112",
]

results_path ./../../../Output_Results/
tl_results_path ./../../../Output_Results/2025_testing


In [4]:
def find_all_TL_models(tl_results_path, Architecture_GROUP_List, target_grp):
    TL_Models_list = {}
    for target_id in target_grp:
        Model_List = {}
        for Architecture_GROUP in Architecture_GROUP_List:
            Model_path = f"{tl_results_path}/{Architecture_GROUP}/diagrams_n_individual_models/{target_id}/"
            if os.path.exists(Model_path) and os.path.isdir(Model_path):
                for file_name in os.listdir(Model_path):
                    if file_name.endswith("keras"):
                        Model_List[file_name] = f"{Model_path}{file_name}"
        print(f"Total number of Models in {target_id} : {len(Model_List)}")
        TL_Models_list[target_id] = Model_List
    return TL_Models_list


TL_Models_list = find_all_TL_models(
    tl_results_path, Architecture_GROUP_List, target_grp
)

Total number of Models in SB-001 : 735
Total number of Models in SB-003 : 735
Total number of Models in SB-008 : 735
Total number of Models in SB-011 : 735
Total number of Models in SB-012 : 735
Total number of Models in SB-022 : 735
Total number of Models in SB-023 : 735
Total number of Models in SB-025 : 735
Total number of Models in SB-028 : 735
Total number of Models in SB-037 : 735
Total number of Models in SB-043 : 735
Total number of Models in SB-056 : 735
Total number of Models in SB-059 : 735
Total number of Models in SB-060 : 735
Total number of Models in SB-070 : 735
Total number of Models in SB-071 : 735
Total number of Models in SB-073 : 735
Total number of Models in SB-078 : 735
Total number of Models in SB-079 : 735
Total number of Models in SB-080 : 735
Total number of Models in SB-082 : 735
Total number of Models in SB-083 : 735
Total number of Models in SB-112 : 735


In [5]:
for target_id in target_grp:
    print(target_id, list(TL_Models_list[target_id].keys())[:1])

SB-001 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-001-1000_IND_FOLD_1.keras']
SB-003 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-003-1000_IND_FOLD_1.keras']
SB-008 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-008-1000_IND_FOLD_1.keras']
SB-011 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-011-1000_IND_FOLD_1.keras']
SB-012 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-012-1000_IND_FOLD_1.keras']
SB-022 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-022-1000_IND_FOLD_1.keras']
SB-023 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-023-1000_IND_FOLD_1.keras']
SB-025 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-025-1000_IND_FOLD_1.keras']
SB-028 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-028-1000_IND_FOLD_1.keras']
SB-037 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-037-1000_IND_FOLD_1.keras']
SB-043 ['source_size_5132_TL_All_Unfrozen_1000_POP_FOLD_4_SB-043-1000_IND_FOLD_1.keras']
SB-056 ['source_size_

In [6]:
source_quantile = "0.2"
source_file_path = f"{execution_path}Input_Data/Og_Data/labeled_normal_data/2all_pids_q={source_quantile}.npy"
display(source_file_path)
data_ndarr = np.load(source_file_path, allow_pickle=True)
print("data_ndarr type :", type(data_ndarr))
data_dict = data_ndarr.item()
print("data_dict type :", type(data_dict))
p_ids = data_dict.keys()
sample_size_dict = {}
print("patients: n =", len(p_ids), end="/n/n")
for p_id in p_ids:
    df = data_dict[p_id]
    df.drop(["id", "date", "am_pef_org", "BMI", "sex", "age"], axis=1, inplace=True)
    df = np.asarray(df).astype(np.float32)
    print(p_id, "shape:", df.shape)
    sample_size_dict[p_id] = df.shape[0]
display(data_dict[p_id].head(5))
display(sample_size_dict)

'../../../Input_Data/Og_Data/labeled_normal_data/2all_pids_q=0.2.npy'

data_ndarr type : <class 'numpy.ndarray'>
data_dict type : <class 'dict'>
patients: n = 26/n/nSB-001 shape: (188, 28)
SB-003 shape: (180, 28)
SB-008 shape: (178, 28)
SB-011 shape: (210, 28)
SB-012 shape: (194, 28)
SB-022 shape: (204, 28)
SB-023 shape: (194, 28)
SB-025 shape: (117, 28)
SB-028 shape: (92, 28)
SB-033 shape: (53, 28)
SB-037 shape: (210, 28)
SB-043 shape: (191, 28)
SB-056 shape: (191, 28)
SB-059 shape: (203, 28)
SB-060 shape: (88, 28)
SB-070 shape: (202, 28)
SB-071 shape: (124, 28)
SB-073 shape: (161, 28)
SB-078 shape: (153, 28)
SB-079 shape: (142, 28)
SB-080 shape: (181, 28)
SB-081 shape: (34, 28)
SB-082 shape: (137, 28)
SB-083 shape: (206, 28)
SB-089 shape: (49, 28)
SB-112 shape: (180, 28)


,income,road_dist,cooking,y_am_pef,tempin,humidin,pm25in,co2in,tempdiffin,humidiffin,pm25diffin,pm10,pm25,o3,no2,co,so2,temp,windsd,humid,varp,dewpt,airp,seap,solrhr,solramnt,grdt,class
0,0.857,0.0,0.167,0.558,0.216,0.513,0.358,0.678,0.562,0.364,0.619,0.390,0.250,0.024,0.587,0.536,0.222,0.349,0.107,0.668,0.174,0.471,0.822,0.819,0.165,0.003,0.206,0.0
1,0.857,0.0,0.167,0.006,0.299,0.521,0.411,0.700,0.436,0.672,0.616,0.446,0.285,0.000,0.783,0.732,0.222,0.473,0.171,0.388,0.190,0.491,0.676,0.670,0.603,0.165,0.383,0.0
2,0.857,0.0,0.167,0.078,0.420,0.713,0.372,0.918,0.469,0.821,0.734,0.466,0.155,0.190,0.239,0.262,0.111,0.474,0.515,0.359,0.197,0.481,0.519,0.518,0.485,0.104,0.346,1.0
3,0.857,0.0,0.167,0.428,0.438,0.468,0.224,0.762,0.560,0.490,0.794,0.211,0.011,0.143,0.196,0.110,0.111,0.404,0.338,0.527,0.175,0.467,0.596,0.596,0.491,0.141,0.339,0.0
4,0.857,0.0,0.167,0.209,0.462,0.385,0.199,0.502,0.612,0.529,0.725,0.250,0.057,0.095,0.348,0.245,0.111,0.366,0.316,0.444,0.141,0.419,0.765,0.762,0.433,0.135,0.281,1.0


{'SB-001': 188,
 'SB-003': 180,
 'SB-008': 178,
 'SB-011': 210,
 'SB-012': 194,
 'SB-022': 204,
 'SB-023': 194,
 'SB-025': 117,
 'SB-028': 92,
 'SB-033': 53,
 'SB-037': 210,
 'SB-043': 191,
 'SB-056': 191,
 'SB-059': 203,
 'SB-060': 88,
 'SB-070': 202,
 'SB-071': 124,
 'SB-073': 161,
 'SB-078': 153,
 'SB-079': 142,
 'SB-080': 181,
 'SB-081': 34,
 'SB-082': 137,
 'SB-083': 206,
 'SB-089': 49,
 'SB-112': 180}

In [ ]:
def process_model(
    target_id,
    TL_Model_filename,
    TL_Model_filepath,
    Model_Type,
    Architecture_GROUP,
    dataset_list,
):
    if Architecture_GROUP in TL_Model_filename:
        try:
            model = load_model(TL_Model_filepath, compile=False)
            tl_model_for_base = Model(
                inputs=model.input,
                outputs=model.layers[-2].output,
            )
            print(
                f"{target_id} - {Model_Type} - {Architecture_GROUP} - {TL_Model_filename}"
            )
            kfold_evaluation_results_list = []
            kfold_confusion_matrix_list = []
            for i in range(number_of_splits):
                print(
                    f"================================================{i}================================================"
                )
                X_train, X_test, Y_train, Y_test = dataset_list[i]
                num_samples = X_train.shape[0]
                num_batches = (num_samples + BATCH_SIZE - 1) // BATCH_SIZE
                feature_extractor_batches = []
                for batch_idx in range(num_batches):
                    start_idx = batch_idx * BATCH_SIZE
                    end_idx = min(start_idx + BATCH_SIZE, num_samples)
                    batch_features = tl_model_for_base.predict(
                        X_train[start_idx:end_idx], verbose=0
                    )
                    feature_extractor_batches.append(batch_features)
                feature_extractor = np.vstack(feature_extractor_batches)
                X_train_for_base = feature_extractor.reshape(
                    feature_extractor.shape[0], -1
                )
                classifier_model = None
                if Model_Type == "NaiveBayes":
                    parameters = {
                        "kernel": [
                            ConstantKernel(1.0, constant_value_bounds="fixed")
                            * RBF(1.0, length_scale_bounds="fixed"),
                            1.0 * RBF(1.0),
                            DotProduct() + WhiteKernel(),
                        ]
                    }
                    nb = sklearn.gaussian_process.GaussianProcessClassifier(
                        random_state=42
                    )
                    GridNB = GridSearchCV(
                        estimator=nb, param_grid=parameters, cv=5, n_jobs=4
                    )
                    GridNB.fit(X_train, Y_train.ravel())
                    classifier_model = GridNB.best_estimator_
                elif Model_Type == "DecisionTree":
                    parameters = {
                        "max_depth": (
                            2,
                            3,
                            4,
                            5,
                            6,
                            7,
                            8,
                            9,
                            10,
                            11,
                            12,
                            13,
                            14,
                            15,
                            16,
                            17,
                            18,
                            19,
                            20,
                            21,
                            22,
                        ),
                        "criterion": ("entropy", "gini"),
                    }
                    Grid_DT_depth = GridSearchCV(
                        tree.DecisionTreeClassifier(), parameters, cv=5, n_jobs=4
                    )
                    Grid_DT_depth.fit(X_train, Y_train.ravel())
                    classifier_model = Grid_DT_depth.best_estimator_
                elif Model_Type == "LogisticRegression":
                    parameters = {
                        "penalty": ["l1", "l2"],
                        "C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
                        "solver": ["liblinear", "lbfgs"],
                    }
                    GridLR = GridSearchCV(
                        sklearn.linear_model.LogisticRegression(),
                        param_grid=parameters,
                        cv=5,
                        n_jobs=4,
                    )
                    GridLR.fit(X_train, Y_train.ravel())
                    classifier_model = GridLR.best_estimator_
                elif Model_Type == "K-NN":
                    parameters = {
                        "n_neighbors": [3, 5, 11, 9],
                        "weights": ["uniform", "distance"],
                        "metric": ["euclidean", "manhattan"],
                    }
                    GridKNN = GridSearchCV(
                        sklearn.neighbors.KNeighborsClassifier(),
                        param_grid=parameters,
                        cv=5,
                        n_jobs=4,
                    )
                    GridKNN.fit(X_train, Y_train.ravel())
                    classifier_model = GridKNN.best_estimator_
                elif Model_Type == "SVM":
                    parameters = {
                        "kernel": ("linear", "poly", "rbf", "sigmoid"),
                        "C": np.arange(1, 42, 10),
                        "degree": np.arange(3, 6),
                        "coef0": np.arange(0.001, 3, 0.5),
                        "gamma": ("auto", "scale"),
                    }
                    SVModel = SVC()
                    GridS = GridSearchCV(SVModel, parameters, cv=5, n_jobs=4)
                    GridS.fit(X_train, Y_train.ravel())
                    classifier_model = GridS.best_estimator_
                classifier_model.fit(X_train_for_base, Y_train.ravel())
                num_test_samples = X_test.shape[0]
                num_test_batches = (num_test_samples + BATCH_SIZE - 1) // BATCH_SIZE
                test_feature_batches = []
                for batch_idx in range(num_test_batches):
                    start_idx = batch_idx * BATCH_SIZE
                    end_idx = min(start_idx + BATCH_SIZE, num_test_samples)
                    batch_features = tl_model_for_base.predict(
                        X_test[start_idx:end_idx], verbose=0
                    )
                    test_feature_batches.append(batch_features)
                X_test_feature = np.vstack(test_feature_batches)
                X_test_feature = X_test_feature.reshape(X_test_feature.shape[0], -1)
                y_test_prediction = classifier_model.predict(X_test_feature)
                TL_Base_results, c_matrix = evaluate(
                    y_test_prediction.reshape(y_test_prediction.shape[0], -1),
                    Y_test,
                )
                kfold_evaluation_results_list.append(TL_Base_results)
                kfold_confusion_matrix_list.append(c_matrix)
            version_tag = f"{TL_Model_filename[:-6]}_{Model_Type}"
            avg_results = (
                pd.DataFrame(kfold_evaluation_results_list, columns=EVALUATION_METRICS)
                .mean()
                .values
            ).reshape(1, 15)
            avg_results_df = pd.DataFrame(avg_results, columns=EVALUATION_METRICS)
            avg_results_df.index = [version_tag]
            return version_tag, avg_results_df
        except Exception as e:
            print(f"Error processing {TL_Model_filename}: {e}")
            return None, None
    return None, None

In [ ]:
def TL_and_Base_Model(
    target_grp,
    tl_results_path,
    TL_Models_list,
    data_dict,
    EVALUATION_METRICS,
    Architecture_GROUP_List,
    Model_Type_list,
    number_of_splits,
    augmentation_algo,
):
    for target_id in target_grp:
        df_evaluation_results = pd.DataFrame(columns=EVALUATION_METRICS)
        CLASS_VAR = "class"
        dataset_list = split_data(
            target=target_id,
            data=data_dict,
            class_var=CLASS_VAR,
            balance=True,
            test_frac=0.2,
            transfer=False,
            kfold_n_splits=number_of_splits,
            augmentation_algo=augmentation_algo,
        )
        for Architecture_GROUP in Architecture_GROUP_List:
            if os.path.exists(f"{tl_results_path}/{Architecture_GROUP}/"):
                os.makedirs(
                    os.path.dirname(
                        f"{tl_results_path}/{Architecture_GROUP}/diagrams_n_TL-Base_models/"
                    ),
                    exist_ok=True,
                )
                output_results_path = (
                    f"{tl_results_path}/{Architecture_GROUP}/diagrams_n_TL-Base_models/"
                )
                TL_Models = TL_Models_list[target_id]
                for Model_Type in Model_Type_list:
                    results = Parallel(n_jobs=min(8, cores_to_use // 4))(
                        delayed(process_model)(
                            target_id,
                            TL_Model_filename,
                            TL_Model_filepath,
                            Model_Type,
                            Architecture_GROUP,
                            dataset_list,
                        )
                        for TL_Model_filename, TL_Model_filepath in TL_Models.items()
                    )
                    for version_tag, avg_results_df in results:
                        if version_tag is not None and avg_results_df is not None:
                            df_evaluation_results = pd.concat(
                                [df_evaluation_results, avg_results_df]
                            )
                evaluation_results_save_path = os.path.join(
                    f"{output_results_path}TL_Base_{Architecture_GROUP}_evaluation_metrics.csv"
                )
                if os.path.isfile(evaluation_results_save_path):
                    df_evaluation_results.to_csv(
                        evaluation_results_save_path, mode="a", header=False
                    )
                else:
                    df_evaluation_results.to_csv(evaluation_results_save_path, mode="a")
                df_evaluation_results = pd.DataFrame(columns=EVALUATION_METRICS)

In [ ]:
TL_and_Base_Model(
    target_grp,
    tl_results_path,
    TL_Models_list,
    data_dict,
    EVALUATION_METRICS,
    Architecture_GROUP_List,
    Model_Type_list,
    number_of_splits,
    augmentation_algo,
)

In [7]:
def segregate_identifier(
    index,
    column_return_type,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    target_grp,
    num_pop_folds,
    num_ind_folds,
    Model_Type_list,
):
    index = str(index).upper()
    if column_return_type == "Pop_Epoch_Size":
        for i in POPULATION_EPOCH_SIZE_LIST:
            if str(i) + "_POP" in index:
                return i
    elif column_return_type == "Ind_Epoch_Size":
        for j in PATIENT_EPOCH_SIZE_LIST:
            if str(j) + "_IND" in index:
                return j
    elif column_return_type == "Patient_ID":
        for i in target_grp:
            if str(i) in index:
                return i
    elif column_return_type == "Pop_Fold":
        for i in [*range(1, num_pop_folds + 1)]:
            if "POP_FOLD_" + str(i) in index:
                return i
    elif column_return_type == "Ind_Fold":
        for i in [*range(1, num_ind_folds + 1)]:
            if "IND_FOLD_" + str(i) in index:
                return i
    elif column_return_type == "Model_Type":
        for i in Model_Type_list:
            if i.upper() in index:
                return i.upper()


def load_csv_to_df(
    file_path,
    data_processing_columns,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    target_grp,
    num_pop_folds,
    num_ind_folds,
    Model_Type_list,
):
    df = pd.read_csv(file_path, index_col=0)
    for i in data_processing_columns:
        df[i] = df.index.map(
            lambda x: segregate_identifier(
                x,
                i,
                POPULATION_EPOCH_SIZE_LIST,
                PATIENT_EPOCH_SIZE_LIST,
                target_grp,
                num_pop_folds,
                num_ind_folds,
                Model_Type_list,
            )
        )
    return df


def resave_eval_df(
    eval_data_type,
    RESULT_DIR,
    Architecture_GROUP,
    bool_remove_target,
    data_processing_columns,
    POPULATION_EPOCH_SIZE_LIST,
    PATIENT_EPOCH_SIZE_LIST,
    target_grp,
    num_pop_folds,
    num_ind_folds,
    Model_Type_list,
):
    if bool_remove_target == True:
        suffix = "_evaluation_metrics_TargetRemoved.csv"
    else:
        suffix = "_evaluation_metrics.csv"
    if eval_data_type == "individual":
        eval_data_path = "/diagrams_n_individual_models/Ind_"
    elif eval_data_type == "population":
        eval_data_path = "/diagrams_n_population_models/Pop_"
    elif eval_data_type == "TL-Base":
        eval_data_path = "/diagrams_n_TL-Base_models/TL_Base_"
    file_path = (
        str(RESULT_DIR)
        + "/"
        + Architecture_GROUP
        + eval_data_path
        + Architecture_GROUP
        + suffix
    )
    if os.path.isfile(file_path):
        temp_unsorted_eval_data_df = load_csv_to_df(
            file_path,
            data_processing_columns,
            POPULATION_EPOCH_SIZE_LIST,
            PATIENT_EPOCH_SIZE_LIST,
            target_grp,
            num_pop_folds,
            num_ind_folds,
            Model_Type_list,
        )

        temp_sorted_eval_data_df = temp_unsorted_eval_data_df.sort_values(
            by=data_processing_columns, ascending=True
        ).drop(columns=data_processing_columns, errors="ignore")

        output_path = os.path.join(file_path)
        temp_sorted_eval_data_df.to_csv(output_path, mode="w")


data_processing_columns = [
    "Patient_ID",
    "Pop_Epoch_Size",
    "Pop_Fold",
    "Ind_Epoch_Size",
    "Ind_Fold",
    "Model_Type",
]
bool_remove_target_list = [False]
eval_data_type_list = ["TL-Base"]


def process_resave(eval_data_type, bool_remove_target, Architecture_GROUP):
    resave_eval_df(
        eval_data_type,
        tl_results_path,
        Architecture_GROUP,
        bool_remove_target,
        data_processing_columns,
        POPULATION_EPOCH_SIZE_LIST,
        PATIENT_EPOCH_SIZE_LIST,
        target_grp,
        num_pop_folds,
        num_ind_folds,
        Model_Type_list,
    )


Parallel(n_jobs=min(4, cores_to_use // 8))(
    delayed(process_resave)(eval_data_type, bool_remove_target, Architecture_GROUP)
    for eval_data_type in eval_data_type_list
    for bool_remove_target in bool_remove_target_list
    for Architecture_GROUP in Architecture_GROUP_List
)
del data_processing_columns
del bool_remove_target_list
del eval_data_type_list